# Seq2Seq 실습 1: 영어-한국어 번역


### 1. 환경 설정 및 라이브러리 설치

* `torch`: 딥러닝 프레임워크
* `torchtext`: 텍스트 처리를 위한 라이브러리
* `spacy`: 영어 토큰화
* `konlpy`: 한국어 토큰화 (Okt 사용)

In [ ]:
# =========================================
# 전체 흐름 요약 (주석으로만 설명)
# - 목적: Colab 환경에서 PyTorch(버전 맞춤), TorchVision, Torchaudio, TorchText, SpaCy, KoNLPy를 설치하고
#         영어 모델(SpaCy) 다운로드, 그리고 KoNLPy가 필요로 하는 Java(OpenJDK 8)를 설치한다.
# - 왜 이렇게 하나?
#   1) 버전 고정: torch와 torchtext 등은 서로 의존성이 맞아야 예제 코드가 오류 없이 돌아감.
#   2) NLP 준비: SpaCy(영어 토큰화/품사), KoNLPy(한국어 형태소 분석) 모두 필요.
#   3) KoNLPy는 Java 기반: 자바 런타임(JDK)이 없으면 KoNLPy가 동작하지 않음.
# - 결과: 이 셀을 성공적으로 실행하면 다음 셀들에서 한국어/영어 텍스트 처리와
#         PyTorch 기반 모델(Seq2Seq 포함)을 바로 실습할 수 있는 환경이 갖춰짐.
# =========================================

# =========================================
# 1) PyTorch 및 관련 패키지 설치 (버전 명시)
# - --upgrade: 현재 설치된 버전보다 낮아도 지정한 버전으로 올리기 위해 사용.
# - torch==2.3.0, torchvision==0.18.0, torchaudio==2.3.0, torchtext==0.18.0:
#   서로 호환되는 릴리즈 조합. Colab/CUDA 환경과 충돌을 줄이고,
#   torchtext 데이터/필드/데이터셋 API 사용 시 오류를 방지.
# - 이렇게 버전을 고정하면 "AttributeError/ImportError" 같은 의존성 문제를 예방.
# =========================================
!pip install --upgrade torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 torchtext==0.18.0

# =========================================
# 2) NLP 라이브러리 설치
# - spacy: 영어 자연어 처리(토큰화, 품사 태깅, 의존 구문 분석 등)에 널리 쓰이는 라이브러리.
# - konlpy: 한국어 형태소 분석(예: Okt, Mecab, Kkma 등 래퍼)을 제공하는 파이썬 패키지.
#   번역/토큰화 파이프라인에서 한국어 텍스트 전처리에 활용.
# =========================================
!pip install spacy konlpy

# =========================================
# 3) SpaCy 영어 소형 모델 다운로드
# - "en_core_web_sm": 소형 영어 모델(빠르고 가벼움). 토큰화/품사 태깅 등에 충분.
# - python -m spacy download: 스크립트 방식으로 모델 리소스를 로컬에 설치.
# - 이후 코드에서 `spacy.load("en_core_web_sm")` 로 바로 사용 가능.
# =========================================
!python -m spacy download en_core_web_sm

# =========================================
# 4) KoNLPy 의존성: 자바(JDK) 설치
# - KoNLPy는 내부적으로 자바 기반 형태소 분석기를 호출하는 구조이므로,
#   시스템에 Java 런타임(JRE/JDK)이 반드시 있어야 함.
# - openjdk-8-jdk: 안정적으로 KoNLPy와 호환되는 버전(8)을 설치.
# - -y: 사용자 확인 없이 자동 설치 진행.
# - 설치 후, Okt 등 대부분의 KoNLPy 기능을 사용할 수 있게 됨.
# =========================================
!apt-get install openjdk-8-jdk -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 37.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 146.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  ca-certificates-java fonts-dejavu-core fonts-dejavu-extra java-common
  libatk-wrapper-java libatk-wrapper-java-jni libgail-common libgail18
  libgtk2.0-0 libgtk2.0-bin libgtk2.0-common libpcsclite1 librsvg2-common
  libxt-dev libxtst6 libxxf86dga1 openjdk-8-jdk-headless

In [ ]:
import torch
import torchtext

print(torch.__version__)
print(torchtext.__version__)

2.3.0+cu121
0.18.0+cpu


In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: Colab 환경에서 한국어 형태소 분석기(KoNLPy)를 안정적으로 쓰기 위해
#         JPype와 Java(JDK)를 정리 후 재설치하는 과정.
# - 왜 필요한가?
#   * KoNLPy는 내부적으로 Java 기반 형태소 분석기를 호출 → Java 런타임 필수
#   * JPype는 Python ↔ Java 연결 다리 역할 → 버전 충돌 시 오류 발생
#   * 따라서 기존 설치를 제거하고, 호환되는 버전으로 재설치해야 안정적
# =========================================

# 1) JPype 관련 패키지 싹 제거
# - Colab 환경에는 jpype, JPype1 등 여러 이름으로 설치된 경우가 있음
# - 버전 충돌을 막기 위해 모두 제거
!pip uninstall -y jpype JPype1 jpype1

# 2) 자바 설치 (Colab용, JDK 17)
# - KoNLPy는 Java 기반이므로 JDK 필요
# - openjdk-17-jdk-headless: GUI 없는 서버 환경용 JDK 17
# - apt-get update: 패키지 목록 최신화
# - apt-get install: 실제 설치 진행
!apt-get -y update
!apt-get -y install openjdk-17-jdk-headless

# 3) JPype1, konlpy 깔끔하게 재설치
# - JPype1==1.4.1: 안정적으로 KoNLPy와 호환되는 버전
# - konlpy: 한국어 형태소 분석 라이브러리 (Okt, Mecab, Kkma 등 제공)
# - 이 과정을 거치면 Python ↔ Java 연결이 정상화되어 KoNLPy 사용 가능
!pip install JPype1==1.4.1
!pip install konlpy

Found existing installation: jpype1 1.6.0
Uninstalling jpype1-1.6.0:
  Successfully uninstalled jpype1-1.6.0
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,289 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,123 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,526 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,436 kB]
Get:12 http://securit

### 2. 데이터 다운로드 및 전처리

[Tatoeba 프로젝트](http://www.manythings.org/anki/)의 소규모 영-한 병렬 코퍼스를 사용

In [ ]:
# 1) 한국어-영어 병렬 데이터 다운로드
# - manythings.org에서 제공하는 Anki용 병렬 코퍼스 (kor-eng.zip)
# - wget: 웹에서 파일 다운로드
!wget http://www.manythings.org/anki/kor-eng.zip

# 2) 압축 해제
# - unzip: 다운로드한 zip 파일을 풀어서 텍스트 데이터 확보
!unzip kor-eng.zip

### 3. 기본 라이브러리 임포트 및 설정

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 한국어-영어 병렬 코퍼스를 다운로드하고 압축을 풀어 학습 데이터로 사용.
# - 이후: PyTorch, TorchText, SpaCy, KoNLPy 등 필요한 라이브러리를 불러와
#         Seq2Seq 번역 모델 학습 환경을 준비한다.
# - 핵심 포인트:
#   1) 데이터셋 준비 (kor-eng.zip)
#   2) 딥러닝 라이브러리 임포트
#   3) 한국어/영어 전처리 도구 준비
#   4) 재현성 확보를 위한 시드 고정
#   5) GPU 사용 여부 확인
# =========================================

# 3) PyTorch 관련 라이브러리 임포트
import torch
import torch.nn as nn              # 신경망 모듈 (레이어, 모델 정의)
import torch.optim as optim        # 최적화 알고리즘 (SGD, Adam 등)
import torch.nn.functional as F    # 활성화 함수, 손실 함수 등

# 4) 데이터 처리 관련 모듈
from torch.utils.data import Dataset, DataLoader   # 커스텀 데이터셋/배치 로더
from torch.nn.utils.rnn import pad_sequence        # 시퀀스 길이 맞추기 (패딩)
from torchtext.vocab import build_vocab_from_iterator  # 단어 사전 생성

# 5) NLP 라이브러리
import spacy                        # 영어 토큰화/품사 태깅
from konlpy.tag import Okt          # 한국어 형태소 분석기 (예: "사과를 먹었다" → ['사과','를','먹었','다'])

# 6) 기타 유틸리티 라이브러리
import random
import math
import time
import io
import os

# 7) 시드 고정
# - 딥러닝 학습에는 무작위 요소가 많음 (가중치 초기화, 데이터 셔플, dropout 등)
# - 같은 코드 + 같은 데이터 + 같은 시드를 주면 같은 결과를 재현 가능
SEED = 1234
random.seed(SEED)                   # 파이썬 random 모듈 시드
torch.manual_seed(SEED)             # PyTorch CPU 시드
torch.cuda.manual_seed(SEED)        # PyTorch GPU 시드
torch.backends.cudnn.deterministic = True  # CUDA 연산을 결정론적으로 실행

# 8) 디바이스 설정
# - GPU(CUDA)가 사용 가능하면 GPU, 아니면 CPU 선택
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 4. 토크나이저 정의

영어는 `spaCy`, 한국어는 `Okt`를 사용해 토크나이저를 정의

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 영어와 한국어 문장을 각각 토큰화할 수 있는 함수를 정의한다.
# - 왜 필요한가?
#   * Seq2Seq 번역 모델은 "문장을 단어 단위(토큰)"로 나눠야 학습 가능.
#   * 영어는 spaCy, 한국어는 KoNLPy(Okt 형태소 분석기)를 사용.
# - 결과: tokenize_en(), tokenize_ko() 함수로 문장을 토큰 리스트로 변환.
# =========================================

import spacy
from konlpy.tag import Okt

# 1) 영어 토크나이저 로드
# - spaCy의 소형 영어 모델(en_core_web_sm)을 불러옴
# - 이 모델은 영어 문장을 토큰화, 품사 태깅, 의존 구문 분석 등을 지원
spacy_en = spacy.load('en_core_web_sm')

# 2) 한국어 토크나이저 로드
# - KoNLPy의 Okt(Open Korean Text) 형태소 분석기
# - 한국어 문장을 형태소 단위로 분리 (예: "사과를 먹었다" → ['사과','를','먹었','다'])
okt = Okt()

# 3) 영어 토큰화 함수 정의
def tokenize_en(text):
    """
    spaCy를 사용한 영어 토크나이저
    - 입력: 문자열 (예: "Hello, my name is Kim.")
    - 출력: 토큰 리스트 (예: ["Hello", ",", "my", "name", "is", "Kim", "."])
    """
    return [tok.text for tok in spacy_en.tokenizer(text)]

# 4) 한국어 토큰화 함수 정의
def tokenize_ko(text):
    """
    Okt를 사용한 한국어 토크나이저
    - 입력: 문자열 (예: "안녕하세요, 제 이름은 김입니다.")
    - 출력: 형태소 리스트 (예: ["안녕", "하", "세요", ",", "제", "이름", "은", "김", "입니다", "."])
    """
    return okt.morphs(text)

# 5) 테스트 실행
# - 영어 문장을 spaCy로 토큰화
print(tokenize_en("Hello, my name is Kim."))
# 결과 예시: ['Hello', ',', 'my', 'name', 'is', 'Kim', '.']

# - 한국어 문장을 Okt로 토큰화
print(tokenize_ko("안녕하세요, 제 이름은 김입니다."))
# 결과 예시: ['안녕', '하', '세요', ',', '제', '이름', '은', '김', '입니다', '.']

### 5. 데이터 로드 및 어휘장(Vocabulary) 구축

다운로드한 `kor.txt` 파일(탭으로 구분)을 읽고, 영어와 한국어 어휘장을 만듦

In [ ]:
import io

# =========================================
# 전체 흐름 설명
# - 목적: 병렬 코퍼스(kor.txt)를 읽어 영어/한국어 단어 사전을 구축한다.
# - 과정:
#   1) 데이터 파일에서 문장을 한 줄씩 읽음
#   2) 영어/한국어 각각 토큰화 함수로 분리
#   3) torchtext의 build_vocab_from_iterator로 단어장(vocab) 생성
#   4) 최소 빈도(min_freq) 이하 단어는 제외
#   5) 특수 토큰(<unk>, <pad>, <sos>, <eos>)을 단어장에 포함
# - 결과: vocab_en, vocab_ko 두 개의 단어장 객체 생성
# =========================================

DATA_PATH = 'kor.txt'   # 병렬 코퍼스 파일 경로
MIN_FREQ = 5            # 단어장에 포함될 최소 빈도 (5번 이상 등장해야 포함)

# 특수 토큰 정의 (인덱스도 함께 지정)
UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<sos>', '<eos>']

# =========================================
# 영어 토큰 제너레이터
# - kor.txt 파일을 한 줄씩 읽음
# - 각 줄은 "영어문장 \t 한국어문장" 구조
# - parts[0]: 영어 문장
# - tokenizer_en으로 토큰화한 결과를 yield (제너레이터로 반환)
# =========================================
def yield_en_tokens(data_path, tokenizer_en):
    with io.open(data_path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')  # 탭으로 분리
            if len(parts) >= 2:               # 최소 두 필드(영어, 한국어)가 있어야 함
                yield tokenizer_en(parts[0])  # 영어 문장을 토큰화해 반환

# =========================================
# 한국어 토큰 제너레이터
# - parts[1]: 한국어 문장
# - tokenizer_ko로 토큰화한 결과를 yield
# =========================================
def yield_ko_tokens(data_path, tokenizer_ko):
    with io.open(data_path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                yield tokenizer_ko(parts[1])  # 한국어 문장을 토큰화해 반환

from torchtext.vocab import build_vocab_from_iterator

# =========================================
# 영어 단어장 구축
# - yield_en_tokens로 생성된 토큰들을 기반으로 vocab 생성
# - min_freq=MIN_FREQ: 최소 5번 이상 등장한 단어만 포함
# - specials: 특수 토큰 추가
# - special_first=True: 특수 토큰을 단어장 앞쪽에 배치
# =========================================
print("Building English vocab...")
vocab_en = build_vocab_from_iterator(
    yield_en_tokens(DATA_PATH, tokenize_en),
    min_freq=MIN_FREQ,
    specials=special_symbols,
    special_first=True
)
vocab_en.set_default_index(UNK_IDX)  # OOV 단어는 <unk> 인덱스로 매핑

# =========================================
# 한국어 단어장 구축
# - yield_ko_tokens로 생성된 토큰들을 기반으로 vocab 생성
# - 동일하게 min_freq, specials 적용
# =========================================
print("Building Korean vocab...")
vocab_ko = build_vocab_from_iterator(
    yield_ko_tokens(DATA_PATH, tokenize_ko),
    min_freq=MIN_FREQ,
    specials=special_symbols,
    special_first=True
)
vocab_ko.set_default_index(UNK_IDX)

# =========================================
# 단어장 크기 출력
# - 영어/한국어 각각 몇 개의 토큰이 포함되었는지 확인
# =========================================
print(f"English Vocab Size: {len(vocab_en)}")
print(f"Korean Vocab Size: {len(vocab_ko)}")

### 6. Dataset 및 DataLoader 정의

PyTorch의 `Dataset`과 `DataLoader`를 사용하여 데이터를 배치 단위로 처리할 수 있도록 준비합니다.

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 병렬 코퍼스(kor.txt)를 PyTorch Dataset 형태로 정의한다.
# - Dataset은 DataLoader와 함께 쓰여서 배치 단위로 데이터를 공급한다.
# - 여기서는 영어 문장과 한국어 문장을 쌍으로 읽어와 저장하고,
#   __getitem__을 통해 인덱스로 접근할 수 있게 한다.
# =========================================

# PyTorch의 Dataset(데이터셋 베이스 클래스)과 DataLoader(배치 생성기) 임포트
from torch.utils.data import Dataset, DataLoader

# PyTorch Dataset을 상속해 영어-한국어 병렬 데이터셋 클래스를 정의
class EngKorDataset(Dataset):
    # 생성자: 데이터 경로, 영어/한국어 vocab, 토크나이저를 받아 내부 상태 초기화
    def __init__(self, data_path, vocab_en, vocab_ko, tokenizer_en, tokenizer_ko):
        # 영어 단어장(vocab) 객체를 멤버 변수로 저장 (토큰 → 인덱스 변환 등에 사용 예정)
        self.vocab_en = vocab_en
        # 한국어 단어장(vocab) 객체 저장
        self.vocab_ko = vocab_ko
        # 영어 토크나이저 함수 저장
        self.tokenizer_en = tokenizer_en
        # 한국어 토크나이저 함수 저장
        self.tokenizer_ko = tokenizer_ko
        # (영어 문장, 한국어 문장) 튜플을 담을 리스트 초기화
        self.data = []

        # 데이터 파일을 UTF-8 인코딩으로 열기 (텍스트 깨짐 방지)
        with io.open(data_path, encoding='utf-8') as f:
            # 파일을 줄 단위로 순회
            for line in f:
                # 양끝 공백/개행 제거 후 탭(\t) 기준으로 분할 → ["영어문장", "한국어문장"]
                parts = line.strip().split('\t')
                # 최소한 영어/한국어 두 필드가 있는 줄만 유효 데이터로 사용
                if len(parts) >= 2:
                    # 리스트에 (영어문장, 한국어문장) 튜플 추가
                    self.data.append((parts[0], parts[1]))

    # 데이터셋의 샘플 개수 반환 (len(dataset) 호출 시 사용)
    def __len__(self):
        # 누적된 문장 쌍의 개수 반환
        return len(self.data)

    # 인덱싱 접근 지원 (dataset[idx]) → 한 샘플(문장 쌍) 반환
    def __getitem__(self, idx):
        # idx 위치의 (영어문장, 한국어문장) 튜플 꺼내기
        eng_text, kor_text = self.data[idx]
        # 원문 문자열 그대로 반환 (토큰화/인덱스 변환은 collate_fn에서 수행 예정)
        return eng_text, kor_text

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

# =========================================
# 전체 흐름 설명
# - 목적: Dataset에서 꺼낸 (영어, 한국어) 문장 쌍을
#         토큰화 → 인덱스 변환 → 특수토큰 추가 → 패딩 으로 정규화해
#         RNN이 받기 좋은 형태 (seq_len, batch_size)로 만드는 collate_fn 정의.
# - 그리고 학습/검증 데이터로 나누고, DataLoader로 배치 생성 테스트까지 수행.
# =========================================

# -----------------------------------------
# 1) 디바이스 설정: CUDA(GPU) 가능하면 GPU, 아니면 CPU
# -----------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")  # 현재 사용 중인 디바이스 출력

# -----------------------------------------
# 2) collate_fn 정의
# - DataLoader가 batch를 만들 때 호출되어
#   개별 샘플들을 하나의 텐서로 합치는 역할
# -----------------------------------------
def collate_fn(batch):
    """
    PyTorch의 DataLoader는 여러 샘플을 묶어 배치를 만들지만,
    NLP에서는 문장 길이가 제각각이므로 그대로는 텐서로 묶기 어려움.
    collate_fn은 각 문장을:
      - 토큰화
      - 인덱스 변환(vocab 사용)
      - 특수 토큰(<sos>, <eos>) 추가
      - 패딩(pad) 처리
    하여 동일한 길이로 맞춰 (seq_len, batch_size) 형태로 반환.
    """

    # 배치에서 들어온 (영어문장, 한국어문장)들을 담을 리스트 초기화
    src_batch, trg_batch = [], []

    # batch는 [(영문, 한글), (영문, 한글), ...] 형태의 리스트
    for src_sample, trg_sample in batch:
        # 1. 토큰화
        # - 영어: 소문자화 후 spaCy 토크나이저 사용
        src_tokens = [tok for tok in tokenize_en(src_sample.lower())]
        # - 한국어: Okt 형태소 분석기 사용
        trg_tokens = [tok for tok in tokenize_ko(trg_sample)]

        # 2. 특수 토큰 추가
        # - 소스: RNN 인코더 종료를 알리는 <eos> 추가
        src_with_eos = src_tokens + ['<eos>']
        # - 타깃: 디코더 입력 시작 <sos> + 종료 <eos> 추가
        trg_with_sos_eos = ['<sos>'] + trg_tokens + ['<eos>']

        # -----------------------------------------
        # 3. 수치화 (토큰 → 인덱스, vocab을 딕셔너리처럼 호출)
        # -----------------------------------------
        # - src_with_eos : 영어 문장을 토큰화한 뒤 마지막에 <eos>를 붙인 리스트
        #   예: ["hello", "world", "<eos>"]
        # - trg_with_sos_eos : 한국어 문장을 토큰화한 뒤 앞뒤에 <sos>, <eos>를 붙인 리스트
        #   예: ["<sos>", "안녕", "세상", "<eos>"]

        # - vocab_en, vocab_ko : 단어장(vocab) 객체
        #   각 토큰(단어)을 고유한 숫자 인덱스로 매핑해줌
        #   예: "hello" → 57, "world" → 132, "<eos>" → 3
        #       "<sos>" → 2, "안녕" → 915, "세상" → 2323, "<eos>" → 3

        # - 리스트 컴프리헨션으로 각 토큰을 vocab에 넣어 인덱스로 변환
        #   src_indices = [vocab_en[token] for token in src_with_eos]
        #   → ["hello", "world", "<eos>"] → [57, 132, 3]
        #   trg_indices = [vocab_ko[token] for token in trg_with_sos_eos]
        #   → ["<sos>", "안녕", "세상", "<eos>"] → [2, 915, 2323, 3]

        # - 만약 단어장이 모르는 단어(OOV)가 들어오면
        #   vocab.set_default_index(UNK_IDX) 설정 덕분에 자동으로 <unk> 인덱스(0)로 매핑됨
        src_indices = [vocab_en[token] for token in src_with_eos]
        trg_indices = [vocab_ko[token] for token in trg_with_sos_eos]

        # 파이썬 리스트를 LongTensor로 변환 (RNN 입력은 long 인덱스 필요)
        src_batch.append(torch.tensor(src_indices, dtype=torch.long))
        trg_batch.append(torch.tensor(trg_indices, dtype=torch.long))

    # 4. 패딩
    # - pad_sequence는 길이가 다른 시퀀스 리스트를 같은 길이로 맞춰 텐서로 변환
    # - 기본(batch_first=False)이므로 결과 형태는 (max_seq_len, batch_size)
    # - padding_value=PAD_IDX: 빈칸을 <pad> 토큰 인덱스로 채움
    src_padded = pad_sequence(src_batch, padding_value=PAD_IDX)  # (src_len, batch)
    trg_padded = pad_sequence(trg_batch, padding_value=PAD_IDX)  # (trg_len, batch)

    # GPU/CPU로 텐서 이동
    return src_padded.to(device), trg_padded.to(device)

# -----------------------------------------
# 3) 데이터셋 분리
# - 전체 데이터에서 학습/검증 세트로 나눔 (여기서는 80%/20%)
# - 테스트 세트는 간단히 검증 세트를 재사용 (실무에서는 별도 분리 권장)
# -----------------------------------------
full_dataset = EngKorDataset(DATA_PATH, vocab_en, vocab_ko, tokenize_en, tokenize_ko)
train_size = int(0.8 * len(full_dataset))          # 학습 샘플 수
valid_size = len(full_dataset) - train_size         # 검증 샘플 수
train_dataset, valid_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, valid_size]
)

# -----------------------------------------
# 4) DataLoader 생성
# - BATCH_SIZE 만큼 샘플을 모아 배치를 구성
# - shuffle=True: 학습 시 데이터 순서를 섞어 일반화에 도움
# - collate_fn=collate_fn: 위에서 정의한 전처리/패딩 로직 적용
# -----------------------------------------
BATCH_SIZE = 128
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# -----------------------------------------
# 5) 데이터로더 테스트
# - 첫 배치를 뽑아 텐서 형태와 차원 확인
# - 출력 형태는 (seq_len, batch_size)로 되어 있어 RNN 입력에 바로 사용 가능
# -----------------------------------------
src_batch, trg_batch = next(iter(train_loader))
print(f"Source batch shape: {src_batch.shape}")  # 예: torch.Size([src_len, batch_size])
print(f"Target batch shape: {trg_batch.shape}")  # 예: torch.Size([trg_len, batch_size])

### 7. Seq2Seq 모델 정의

#### 7.1. Encoder (인코더)

인코더는 입력 문장(영어)을 받아 하나의 '컨텍스트 벡터'(Context Vector)로 압축

In [ ]:
import torch.nn as nn

# =========================================
# 전체 흐름 설명
# - 목적: Seq2Seq 모델의 인코더(Encoder)를 정의한다.
# - 인코더는 입력 문장을 RNN(GRU)으로 읽고,
#   마지막 hidden state를 "컨텍스트 벡터"로 반환한다.
# - 이 컨텍스트 벡터는 디코더가 번역을 시작할 때 초기 상태로 사용된다.
# =========================================

# nn.Module을 상속받아 PyTorch 신경망 모듈로 정의
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        # 부모 클래스 초기화
        super().__init__()
        # 은닉 상태 차원 크기 저장
        self.hid_dim = hid_dim
        # RNN 층 개수 저장
        self.n_layers = n_layers

        # 1. 임베딩 층
        # - input_dim: 단어 사전 크기 (vocab size)
        # - emb_dim: 각 단어를 몇 차원 벡터로 표현할지
        # - 역할: 단어 인덱스를 밀집 벡터로 변환
        self.embedding = nn.Embedding(input_dim, emb_dim)

        # 2. RNN 층 (GRU 사용)
        # - 입력: emb_dim 크기의 벡터
        # - 출력: hid_dim 크기의 은닉 상태
        # - n_layers: GRU 층을 몇 겹으로 쌓을지
        # - dropout: 층 사이에 드롭아웃 적용
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)

        # 3. 드롭아웃 층
        # - 학습 시 일부 뉴런을 무작위로 꺼서 과적합 방지
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: (src_len, batch_size)
        # - 입력 문장(토큰 인덱스 시퀀스)

        # 1. 임베딩
        # - 단어 인덱스를 emb_dim 크기의 벡터로 변환
        # - 드롭아웃 적용
        embedded = self.dropout(self.embedding(src))
        # embedded: (src_len, batch_size, emb_dim)

        # 2. RNN 통과
        # - GRU는 두 가지를 반환:
        #   outputs: 모든 time-step의 top layer hidden state
        #   hidden: 마지막 time-step의 모든 layer hidden state
        outputs, hidden = self.rnn(embedded)

        # outputs: (src_len, batch_size, hid_dim * n_directions)
        # hidden: (n_layers * n_directions, batch_size, hid_dim)

        # 3. 컨텍스트 벡터 반환
        # - 여기서는 outputs는 사용하지 않고,
        #   마지막 hidden state만 디코더로 전달
        return hidden

#### 7.2. Decoder (디코더)

디코더는 인코더의 컨텍스트 벡터와 이전 타임스텝의 예측 단어를 받아 다음 단어를 예측

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: Seq2Seq 모델의 디코더(Decoder)를 정의한다.
# - 디코더는 인코더가 만든 컨텍스트 벡터(hidden state)를 받아
#   매 time-step마다 다음 단어를 예측한다.
# - 구조: 임베딩 → GRU → Linear → 출력 단어 확률
# =========================================

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        # 부모 클래스 초기화
        super().__init__()
        # 출력 단어장 크기 저장 (예측해야 할 단어 개수)
        self.output_dim = output_dim
        # 은닉 상태 차원 크기 저장
        self.hid_dim = hid_dim
        # RNN 층 개수 저장
        self.n_layers = n_layers

        # 1. 임베딩 층
        # - 입력: 단어 인덱스
        # - 출력: emb_dim 크기의 벡터
        self.embedding = nn.Embedding(output_dim, emb_dim)

        # 2. RNN 층 (GRU 사용)
        # - 인코더와 동일한 파라미터(hid_dim, n_layers, dropout)를 가져야 함
        # - 입력: 임베딩 벡터
        # - 출력: 은닉 상태(hidden state)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)

        # 3. Fully Connected (Linear) 층
        # - GRU의 hidden state를 받아 어휘장 크기의 벡터로 변환
        # - 각 단어에 대한 점수(logit)를 출력
        self.fc_out = nn.Linear(hid_dim, output_dim)

        # 드롭아웃 층 (과적합 방지)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden):
        # input: (batch_size) - 현재 time-step의 입력 단어 인덱스
        # hidden: (n_layers, batch_size, hid_dim) - 이전 time-step의 hidden state

        # 1. 입력 단어 임베딩
        # - GRU는 (seq_len, batch_size) 형태를 요구하므로
        #   (batch_size) → (1, batch_size)로 차원 확장
        input = input.unsqueeze(0)
        # input: (1, batch_size)

        # - 임베딩 후 드롭아웃 적용
        embedded = self.dropout(self.embedding(input))
        # embedded: (1, batch_size, emb_dim)

        # 2. RNN 통과
        # - GRU에 임베딩 벡터와 이전 hidden state를 넣어 새로운 hidden state 계산
        output, hidden = self.rnn(embedded, hidden)
        # output: (1, batch_size, hid_dim)
        # hidden: (n_layers, batch_size, hid_dim)

        # 3. 예측 (Linear 층)
        # - GRU 출력(output)을 Linear 층에 통과시켜 단어장 크기의 벡터 생성
        # - squeeze(0): seq_len 차원(=1)을 제거 → (batch_size, hid_dim)
        prediction = self.fc_out(output.squeeze(0))
        # prediction: (batch_size, output_dim) - 각 단어에 대한 점수(logit)

        # 반환: 현재 step의 예측 결과와 새로운 hidden state
        return prediction, hidden

#### 7.3. Seq2Seq (모델 결합)

인코더와 디코더를 결합하고, 'Teacher Forcing'을 구현합니다.

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 인코더와 디코더를 연결해 Seq2Seq 전체 모델을 정의한다.
# - 인코더는 입력 문장을 컨텍스트 벡터(hidden state)로 압축하고,
#   디코더는 이 벡터를 받아 매 time-step마다 다음 단어를 예측한다.
# - Teacher Forcing 기법을 사용해 학습 안정성을 높인다.
# =========================================

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        # 부모 클래스 초기화
        super().__init__()
        # 인코더와 디코더 객체 저장
        self.encoder = encoder
        self.decoder = decoder
        # 학습/추론에 사용할 디바이스(CPU/GPU)
        self.device = device

        # 인코더와 디코더의 hidden dim, layer 수가 동일해야 연결 가능
        assert encoder.hid_dim == decoder.hid_dim, "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, "Encoder and decoder must have same number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):
        # src: (src_len, batch_size) - 인코더에 들어갈 입력 문장(토큰 인덱스 시퀀스)
        # trg: (trg_len, batch_size) - 디코더가 학습할 때 참고할 목표 문장(정답 시퀀스)
        # teacher_forcing_ratio: 학습 시 디코더가 정답 토큰을 얼마나 자주 사용할지 결정하는 비율
        #   (예: 0.5 → 절반은 정답 토큰을, 절반은 디코더가 예측한 토큰을 다음 입력으로 사용)

        # 배치 크기(batch_size): 한 번에 학습하는 문장 개수
        # trg.shape[1] → 두 번째 차원(batch_size)을 가져옴
        batch_size = trg.shape[1]

        # 타깃 문장 길이(trg_len): 목표 문장의 토큰 개수
        # trg.shape[0] → 첫 번째 차원(seq_len)을 가져옴
        trg_len = trg.shape[0]

        # 타깃 단어장 크기(trg_vocab_size): 디코더가 출력해야 하는 클래스 개수(= 단어장 크기)
        # self.decoder.output_dim → 디코더의 출력 차원, 즉 단어장 크기
        trg_vocab_size = self.decoder.output_dim

        # 디코더 출력 결과를 저장할 텐서 초기화
        # 크기: (trg_len, batch_size, vocab_size)
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        # 1. 인코더
        # 입력 문장을 인코더에 넣어 컨텍스트 벡터(hidden state) 생성
        hidden = self.encoder(src)

        # 2. 디코더
        # 디코더의 첫 입력은 <sos> 토큰 (시작 토큰)
        input = trg[0, :]  # (batch_size)

        # trg_len 만큼 반복 (i=0은 <sos>이므로 i=1부터 시작)
        # - 디코더는 한 단어씩 순차적으로 예측을 생성하므로,
        #   목표 문장 길이(trg_len)만큼 루프를 돌면서 단어를 하나씩 생성함
        # - i=0 위치에는 <sos> 토큰이 들어가 있으므로 실제 예측은 i=1부터 시작

        for t in range(1, trg_len):
            # 현재 입력 단어(input)와 이전 hidden state를 디코더에 전달
            # → 디코더는 다음 단어를 예측하고 새로운 hidden state를 반환
            output, hidden = self.decoder(input, hidden)

            # 현재 step에서 나온 예측 결과를 outputs[t]에 저장
            # outputs는 전체 문장 길이만큼의 예측 결과를 담는 텐서
            outputs[t] = output

            # Teacher Forcing 여부 결정
            # - teacher_forcing_ratio(예: 0.5)에 따라 일정 확률로 정답 단어를 다음 입력으로 사용
            # - random.random()은 0~1 사이 난수를 반환하므로,
            #   teacher_forcing_ratio보다 작으면 teacher forcing을 적용
            teacher_force = random.random() < teacher_forcing_ratio

            # -----------------------------------------
            # 모델 예측 결과 중 가장 확률이 높은 단어 인덱스를 top1으로 선택
            # -----------------------------------------
            # - 디코더는 매 time-step마다 "다음 단어가 무엇일지"를 예측함
            # - output은 (batch_size, vocab_size) 형태의 텐서로 반환됨
            #   → 각 배치마다 vocab_size(단어장 크기)만큼의 확률 값이 들어 있음
            #   → 예: vocab_size=5라면 [0.1, 0.05, 0.7, 0.1, 0.05] 이런 식으로 확률 분포가 나옴
            #
            # - argmax(1)은 두 번째 차원(vocab 차원)에서 가장 큰 값을 가진 인덱스를 뽑음
            #   → 즉, "가장 확률이 높은 단어의 인덱스"를 선택하는 것
            #
            # - 예시:
            #   output = [[0.1, 0.05, 0.7, 0.1, 0.05],   # 배치 1의 예측 확률
            #             [0.2, 0.3, 0.1, 0.35, 0.05]]  # 배치 2의 예측 확률
            #
            #   output.argmax(1) → tensor([2, 3])
            #   → 배치 1에서는 인덱스 2(확률 0.7)가 가장 큼
            #   → 배치 2에서는 인덱스 3(확률 0.35)가 가장 큼
            #
            # - 따라서 top1은 "각 배치에서 모델이 가장 자신 있게 예측한 단어 인덱스"를 담은 텐서
            #   이 값은 teacher forcing 여부에 따라 다음 step의 입력으로 사용될 수 있음

            top1 = output.argmax(1)

            # 다음 step의 입력 단어 결정
            # - teacher forcing이면 실제 정답 단어(trg[t])를 사용
            # - 아니면 모델이 방금 예측한 단어(top1)를 사용
            input = trg[t] if teacher_force else top1

        # 루프가 끝나면 전체 출력 결과(outputs)를 반환
        # outputs에는 각 time-step마다 디코더가 예측한 단어 분포가 저장되어 있음
        return outputs

### 8. 모델 학습

#### 8.1. 하이퍼파라미터 및 모델 초기화

In [ ]:
import torch.optim as optim
import torch.nn as nn   # CrossEntropyLoss 사용을 위해 임포트
import torch            # CUDA 사용 여부 확인, 시드 고정 등에 필요

# =========================================
# 전체 흐름 설명
# - 목적: Seq2Seq 모델을 인스턴스화하고 학습 준비를 한다.
# - 과정:
#   1) 모델 파라미터 설정 (임베딩 차원, hidden 차원, dropout 등)
#   2) 인코더/디코더 인스턴스 생성 후 Seq2Seq 모델로 결합
#   3) 모델 파라미터 수 확인
#   4) 옵티마이저(Adam) 정의
#   5) 손실 함수(CrossEntropyLoss) 정의, <pad> 토큰 무시 설정
# =========================================

# -----------------------------------------
# Seq2Seq 모델에서 사용하는 주요 하이퍼파라미터 정의
# -----------------------------------------

# INPUT_DIM = len(vocab_en)
# - 인코더 입력 차원 = 영어 단어장 크기
# - 즉, 영어 문장을 토큰화했을 때 가능한 단어/서브워드 개수
# - Embedding 레이어에서 "단어 인덱스 → 벡터"로 변환할 때 필요한 크기

# OUTPUT_DIM = len(vocab_ko)
# - 디코더 출력 차원 = 한국어 단어장 크기
# - 즉, 디코더가 예측해야 하는 단어의 클래스 개수
# - 최종 Linear 레이어에서 softmax를 통해 vocab 크기만큼 확률 분포를 출력

# ENC_EMB_DIM = 256
# - 인코더 임베딩 차원
# - 영어 단어 인덱스를 256차원 벡터로 변환
# - 예: "hello"(인덱스 57) → [0.12, -0.33, ..., 0.87] (길이 256)

# DEC_EMB_DIM = 256
# - 디코더 임베딩 차원
# - 한국어 단어 인덱스를 256차원 벡터로 변환
# - 인코더와 디코더 임베딩 차원은 같아도 되고 달라도 됨 (여기서는 동일하게 설정)

# HID_DIM = 512
# - RNN(GRU/LSTM)의 hidden state 차원
# - 인코더와 디코더 내부에서 문맥을 저장하는 벡터 크기
# - 값이 클수록 모델 용량이 커지고 더 많은 문맥을 담을 수 있지만 연산량도 증가

# N_LAYERS = 2
# - RNN 레이어 수
# - 인코더와 디코더가 각각 2층 GRU/LSTM으로 구성됨
# - 층을 쌓으면 더 복잡한 패턴을 학습할 수 있음

# ENC_DROPOUT = 0.5
# - 인코더 드롭아웃 비율
# - 학습 시 50% 확률로 뉴런을 무작위로 꺼서 과적합을 방지

# DEC_DROPOUT = 0.5
# - 디코더 드롭아웃 비율
# - 디코더에서도 동일하게 50% 확률로 뉴런을 꺼서 일반화 성능 향상
INPUT_DIM = len(vocab_en)
OUTPUT_DIM = len(vocab_ko)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

# -----------------------------------------
# 2) 인코더/디코더 인스턴스 생성 후 Seq2Seq 모델 결합
# - Encoder와 Decoder를 각각 초기화
# - Seq2Seq 클래스에 전달해 전체 모델 구성
# - .to(device): GPU/CPU로 모델 이동
# -----------------------------------------
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# -----------------------------------------
# 3) 모델 파라미터 수 확인
# - 학습 가능한 파라미터 수를 출력해 모델 크기 확인
# -----------------------------------------
def count_parameters(model):
    # requires_grad=True인 파라미터만 합산
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

# -----------------------------------------
# 4) 옵티마이저 정의
# - Adam: 학습률 자동 조정, 빠른 수렴
# - model.parameters(): 모델의 모든 학습 가능한 파라미터 업데이트
# -----------------------------------------
optimizer = optim.Adam(model.parameters())

# -----------------------------------------
# 5) 손실 함수 정의
# - CrossEntropyLoss: 분류 문제에서 자주 사용
# - ignore_index=PAD_IDX: <pad> 토큰은 손실 계산에서 제외
#   (패딩은 실제 단어가 아니므로 학습에 영향을 주지 않게 함)
# -----------------------------------------
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

The model has 6,532,086 trainable parameters


#### 8.2. 학습 및 평가 함수 정의

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 한 epoch 동안 모델을 학습시키는 함수 정의
# - 과정:
#   1) 모델을 학습 모드로 전환
#   2) 배치 단위로 데이터(iterator) 순회
#   3) 순전파 → 손실 계산 → 역전파 → Gradient Clipping → 파라미터 업데이트
#   4) 배치별 손실을 누적해 평균 손실 반환
# =========================================

def train(model, iterator, optimizer, criterion, clip):
    model.train()  # 학습 모드 (Dropout, BatchNorm 등이 학습 모드로 동작)
    epoch_loss = 0 # 한 epoch 동안의 손실 누적 변수

    # iterator: DataLoader가 제공하는 (src, trg) 배치들
    for i, (src, trg) in enumerate(iterator):
        # 0. 이전 배치에서 계산된 gradient 초기화
        optimizer.zero_grad()

        # 1. 모델 순전파
        # - teacher_forcing_ratio=0.5: 50% 확률로 정답 단어를 디코더 입력으로 사용
        output = model(src, trg, 0.5)

        # output: (trg_len, batch_size, output_dim)
        # trg:    (trg_len, batch_size)

        # output_dim: 단어장 크기 (예측해야 할 클래스 개수)
        output_dim = output.shape[-1]

        # 2. 손실 계산
        # - CrossEntropyLoss는 (N, C) 입력과 (N) 타겟을 기대
        # - <sos> 토큰은 예측 대상이 아니므로 [1:]부터 사용
        output = output[1:].view(-1, output_dim) # ( (trg_len-1)*batch_size, output_dim )
        trg = trg[1:].view(-1)                   # ( (trg_len-1)*batch_size )

        # - 모델 출력과 정답을 비교해 손실 계산
        loss = criterion(output, trg)

        # 3. 역전파 (Backpropagation)
        # - 손실을 기준으로 gradient 계산
        loss.backward()

        # 4. Gradient Clipping
        # - 기울기 폭주 방지: clip 값 이상으로 커지지 않도록 잘라냄
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # 5. 파라미터 업데이트
        optimizer.step()

        # 6. 손실 누적
        epoch_loss += loss.item()

    # 전체 배치 평균 손실 반환
    return epoch_loss / len(iterator)

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 학습된 모델을 평가 모드로 실행해 검증/테스트 데이터셋에 대한 평균 손실을 계산한다.
# - 과정:
#   1) 모델을 평가 모드로 전환 (Dropout 등 비활성화)
#   2) 그래디언트 계산을 끄고 데이터셋을 순회
#   3) 순전파 실행 (Teacher Forcing 비활성화)
#   4) 손실 계산 (CrossEntropyLoss, <sos> 제외)
#   5) 배치별 손실을 누적해 평균 손실 반환
# =========================================

def evaluate(model, iterator, criterion):
    model.eval()  # 평가 모드 (Dropout, BatchNorm 등이 평가 모드로 동작)
    epoch_loss = 0  # 한 epoch 동안의 손실 누적 변수

    # 평가 시에는 파라미터 업데이트가 필요 없으므로 gradient 계산 비활성화
    with torch.no_grad():
        # DataLoader에서 배치 단위로 데이터 순회
        for i, (src, trg) in enumerate(iterator):

            # 1. 모델 순전파
            # - 평가 시에는 Teacher Forcing을 사용하지 않음 (teacher_forcing_ratio=0)
            output = model(src, trg, 0)

            # output: (trg_len, batch_size, output_dim)
            # trg:    (trg_len, batch_size)

            # 2. 손실 계산
            # - CrossEntropyLoss는 (N, C) 입력과 (N) 타겟을 기대
            # - <sos> 토큰은 예측 대상이 아니므로 [1:]부터 사용
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)  # ( (trg_len-1)*batch_size, output_dim )
            trg = trg[1:].view(-1)                   # ( (trg_len-1)*batch_size )

            # - 손실 함수로 출력과 정답 비교
            loss = criterion(output, trg)

            # - 손실 누적
            epoch_loss += loss.item()

    # 전체 배치 평균 손실 반환
    return epoch_loss / len(iterator)

#### 8.3. 실제 학습 실행 : 약 2분 시간 걸림


In [ ]:
import time
import math
import torch
import random

# =========================================
# 전체 흐름 설명
# - 목적: 학습 루프를 실행해 Seq2Seq 모델을 여러 epoch 동안 학습 및 검증한다.
# - 과정:
#   1) epoch 반복
#   2) train()으로 학습 손실 계산
#   3) evaluate()로 검증 손실 계산
#   4) 시간 측정 및 출력
#   5) 가장 좋은(valid loss가 최소인) 모델을 저장
# - 출력: 각 epoch마다 학습/검증 손실과 Perplexity(PPL)를 출력
# =========================================

# 학습 epoch 수
N_EPOCHS = 50
# Gradient Clipping 값
CLIP = 1
# 가장 좋은 검증 손실을 저장할 변수 (초기값: 무한대)
best_valid_loss = float('inf')

# -----------------------------------------
# 시간 측정 함수
# - 시작/종료 시간을 받아 경과 시간(분, 초) 계산
# -----------------------------------------
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

# -----------------------------------------
# 학습 루프
# -----------------------------------------
for epoch in range(N_EPOCHS):
    start_time = time.time()  # epoch 시작 시간 기록

    # 1. 학습 손실 계산
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)

    # 2. 검증 손실 계산
    valid_loss = evaluate(model, valid_loader, criterion)

    end_time = time.time()  # epoch 종료 시간 기록
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    # 3. 가장 좋은 모델 저장
    # - 검증 손실(valid_loss)이 이전보다 낮으면 모델 파라미터 저장
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'seq2seq-model.pt')

    # 4. 결과 출력
    # - Train Loss / Valid Loss
    # - Perplexity(PPL): exp(loss), 낮을수록 모델이 더 좋은 번역을 함
    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Epoch: 01 | Time: 0m 7s
	Train Loss: 0.959 | Train PPL:   2.608
	 Val. Loss: 4.267 |  Val. PPL:  71.301
Epoch: 02 | Time: 0m 6s
	Train Loss: 0.949 | Train PPL:   2.584
	 Val. Loss: 4.148 |  Val. PPL:  63.307
Epoch: 03 | Time: 0m 7s
	Train Loss: 0.918 | Train PPL:   2.503
	 Val. Loss: 4.195 |  Val. PPL:  66.364
Epoch: 04 | Time: 0m 6s
	Train Loss: 0.872 | Train PPL:   2.393
	 Val. Loss: 4.269 |  Val. PPL:  71.468
Epoch: 05 | Time: 0m 7s
	Train Loss: 0.820 | Train PPL:   2.270
	 Val. Loss: 4.332 |  Val. PPL:  76.128
Epoch: 06 | Time: 0m 7s
	Train Loss: 0.775 | Train PPL:   2.170
	 Val. Loss: 4.347 |  Val. PPL:  77.266
Epoch: 07 | Time: 0m 7s
	Train Loss: 0.785 | Train PPL:   2.191
	 Val. Loss: 4.329 |  Val. PPL:  75.877
Epoch: 08 | Time: 0m 7s
	Train Loss: 0.733 | Train PPL:   2.081
	 Val. Loss: 4.403 |  Val. PPL:  81.656
Epoch: 09 | Time: 0m 7s
	Train Loss: 0.749 | Train PPL:   2.115
	 Val. Loss: 4.411 |  Val. PPL:  82.346
Epoch: 10 | Time: 0m 6s
	Train Loss: 0.707 | Train PPL:   2.028


### 9. 결과: 모델 추론 (번역)

학습된 모델을 사용하여 새로운 영어 문장을 한국어로 번역하는 함수를 만듭니다.

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 학습된 Seq2Seq 모델을 사용해 영어 문장을 한국어로 번역(추론)한다.
# - 과정:
#   1) 입력 문장을 토큰화 및 인덱스 변환
#   2) 인코더로 컨텍스트 벡터(hidden state) 생성
#   3) 디코더를 <sos> 토큰부터 시작해 반복 실행
#   4) 가장 확률 높은 단어를 예측해 결과 시퀀스에 추가
#   5) <eos> 토큰을 만나면 종료
#   6) 인덱스를 단어로 변환해 최종 번역 문장 반환
# =========================================

def translate_sentence(sentence, model, vocab_en, vocab_ko, tokenizer_en, device, max_len=50):
    model.eval()  # 평가 모드로 전환 (Dropout, BatchNorm 등 학습 시에만 쓰이는 기능 비활성화)

    # -----------------------------------------
    # 1. 입력 문장 토큰화 및 전처리
    # -----------------------------------------
    # - 영어 문장을 소문자로 변환 후 토크나이저로 단어 단위 분리
    tokens = [token.lower() for token in tokenizer_en(sentence)]
    # - 문장 끝을 알리는 <eos> 토큰 추가
    tokens = tokens + ['<eos>']
    # - 각 토큰을 영어 vocab에서 대응되는 인덱스로 변환
    src_indexes = [vocab_en[token] for token in tokens]

    # - 리스트 형태 (src_len,) → 텐서 형태 (src_len, 1)로 변환
    #   unsqueeze(1)은 배치 차원을 추가해 "문장 1개짜리 배치"로 만듦
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    # -----------------------------------------
    # 2. 인코더
    # -----------------------------------------
    # - 입력 문장을 인코더에 넣어 hidden state(컨텍스트 벡터)를 생성
    # - torch.no_grad(): 추론 모드, 그래디언트 계산 비활성화 (메모리/속도 절약)
    with torch.no_grad():
        hidden = model.encoder(src_tensor)

    # -----------------------------------------
    # 3. 디코더
    # -----------------------------------------
    # - 디코더는 <sos> 토큰부터 시작해서 한 단어씩 예측을 이어감
    # - trg_indexes 리스트에 예측된 단어 인덱스를 순차적으로 저장
    trg_indexes = [SOS_IDX]

    # - 최대 max_len 길이까지 반복 (너무 길어지는 것 방지)
    for i in range(max_len):
        # 현재 마지막 단어 인덱스를 텐서로 변환
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        # (1,) → (1,1) 형태로 변환 (seq_len=1, batch_size=1)
        trg_tensor = trg_tensor.unsqueeze(0)

        # 디코더 실행 (teacher forcing 없음 → 모델 예측만 사용)
        with torch.no_grad():
            # 디코더는 현재 입력 단어와 hidden state를 받아 다음 단어 확률 분포를 출력
            output, hidden = model.decoder(trg_tensor.squeeze(0), hidden)

        # output: (batch_size, vocab_size) 형태의 확률 분포
        # argmax(1): 각 배치에서 가장 확률이 높은 단어 인덱스를 선택
        pred_token = output.argmax(1).item()
        # 선택된 단어 인덱스를 결과 리스트에 추가
        trg_indexes.append(pred_token)

        # 만약 <eos> 토큰을 만나면 번역 종료
        if pred_token == EOS_IDX:
            break

    # -----------------------------------------
    # 4. 인덱스를 단어로 변환
    # -----------------------------------------
    # - vocab_ko.get_itos(): 인덱스를 실제 한국어 토큰으로 변환
    trg_tokens = [vocab_ko.get_itos()[i] for i in trg_indexes]

    # - <sos>와 <eos>는 번역 결과에서 제외하고 최종 문장 반환
    return " ".join(trg_tokens[1:-1])

In [ ]:
# =========================================
# 전체 흐름 설명
# - 목적: 학습된 Seq2Seq 모델을 불러와 실제 번역 결과를 확인한다.
# - 과정:
#   1) 저장된 모델 파라미터 로드
#   2) 검증 데이터셋(valid_dataset)에서 샘플 몇 개를 뽑아 번역 테스트
#   3) 임의의 영어 문장을 입력해 모델 번역 결과 확인
# - 출력: 원본 영어 문장, 정답 한국어 문장, 모델 번역 결과
# =========================================

# 1) 학습된 모델 로드
# - torch.load로 저장된 state_dict 불러오기
# - model.load_state_dict로 모델 파라미터 복원
model.load_state_dict(torch.load('seq2seq-model.pt'))

# 2) 검증 데이터셋에서 샘플 번역 테스트
print("--- 번역 테스트 ---")
for i in range(5):
    # valid_dataset[i]는 (영어문장, 한국어문장) 튜플 반환
    src, trg = valid_dataset[i]

    # translate_sentence 함수로 영어 문장을 한국어로 번역
    translation = translate_sentence(src, model, vocab_en, vocab_ko, tokenize_en, device)

    # 결과 출력
    print(f"원본 (Eng): {src}")       # 영어 원문
    print(f"정답 (Kor): {trg}")       # 정답 한국어 문장
    print(f"번역 (Model): {translation}\n")  # 모델 번역 결과

# 3) 임의의 문장 테스트
# - 학습 데이터셋에 없는 문장을 넣어 모델의 일반화 성능 확인
custom_sentence = "A man is playing a guitar."
translation = translate_sentence(custom_sentence, model, vocab_en, vocab_ko, tokenize_en, device)
print(f"원본 (Eng): {custom_sentence}")
print(f"번역 (Model): {translation}\n")

custom_sentence = "I know it"
translation = translate_sentence(custom_sentence, model, vocab_en, vocab_ko, tokenize_en, device)
print(f"원본 (Eng): {custom_sentence}")
print(f"번역 (Model): {translation}\n")

custom_sentence = "Hello"
translation = translate_sentence(custom_sentence, model, vocab_en, vocab_ko, tokenize_en, device)
print(f"원본 (Eng): {custom_sentence}")
print(f"번역 (Model): {translation}\n")

--- 번역 테스트 ---
원본 (Eng): I want to get a job that mixes work and play.
정답 (Kor): 나는 일과 놀이가 섞인 직업을 갖고 싶다.
번역 (Model): 나 는 <unk> <unk> 를 할 수 있는 직업 을 갖고 싶어요 .

원본 (Eng): Children need to be fed.
정답 (Kor): 어린이들은 먹어야만 한다.
번역 (Model): 시작 해야 한다 .

원본 (Eng): Fantastic!
정답 (Kor): 끝내주네!
번역 (Model): !

원본 (Eng): Many people eat turkey on Christmas Day.
정답 (Kor): 많은 사람들은 크리스마스 때 칠면조를 먹어.
번역 (Model): 어떤 사람 들 이 톰 으로 <unk> 하게 했다 .

원본 (Eng): I wanted to do something nice for you.
정답 (Kor): 뭔가 너한테 좋은 것을 해주고 싶었어.
번역 (Model): 당신 에게 줄 수 <unk> <unk> .

원본 (Eng): A man is playing a guitar.
번역 (Model): 좋은 은 체스 인 다 .

원본 (Eng): I know it
번역 (Model): 알 고 있어 .

원본 (Eng): Hello
번역 (Model): 했어 .

